# Phases: Where The Selection Rules Come From

Every textbook gives the reflection conditions as rules to memorize. Body-centred: $h+k+l$ even.
Face-centred: $h$, $k$, $l$ all the same parity. Diamond: like fcc, *but* $(222)$ is missing. Hexagonal
close-packed: absent when $h + 2k = 3n$ and $l$ is odd.

They are not rules. They are the zeros of one sum:

$$F_{hkl} = \sum_j f_j\, e^{2\pi i (h x_j + k y_j + l z_j)},$$

taken over the atoms in the cell. Change where the atoms sit and the pattern of zeros changes with
them. This tutorial computes that sum from the *actual atomic coordinates* of four pinned structures
and watches all four classical rules fall out, including the two that a centring test cannot produce:
the diamond $(222)$ and the hcp $(0001)$.

That distinction matters for more than teaching. PyTex has a fast centring predicate,
`ReflectionCondition`, and it is *not* the selection rule — it cannot be, because a screw axis
extinguishes reflections in a primitive lattice where centring forbids nothing. Knowing which of the
two you are using is the difference between a simulated pattern that is right and one that shows
reflections the crystal does not produce.

## Learning goals

1. What are the four things a `Phase` carries, and which question does each answer?
2. How do the bcc, fcc, diamond and hcp selection rules follow from one lattice sum?
3. Why can a centring predicate not be the whole selection rule?
4. What does the conventional-cell site count mean, and why is it not the formula unit?
5. What does a pinned, hash-verified fixture buy over "download the CIF"?

## 0. Setup

Four structures from the pinned fixture corpus: body-centred iron, face-centred nickel, diamond, and
hexagonal close-packed zirconium. Each carries its own atomic basis, which is the only input the
selection rules need.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("ignore", message="Issues encountered while parsing CIF")
warnings.filterwarnings("ignore", message="No _symmetry_equiv_pos_as_xyz")

from pytex import (
    FrameDomain,
    MillerPlaneSet,
    ReferenceFrame,
    get_phase_fixture,
    list_phase_fixtures,
    metric_tensor,
    reciprocal_metric_tensor,
)
from pytex.diffraction.kinematic import centering_allowed_mask
from pytex.diffraction.physics import ReflectionCondition, phase_centering_is_declared
from pytex.diffraction.scattering import electron_structure_factor_angstrom

np.set_printoptions(precision=4, suppress=True)

CRYSTAL = ReferenceFrame("crystal", FrameDomain.CRYSTAL, ("a", "b", "c"))
PHASES = {
    fixture_id: get_phase_fixture(fixture_id).load_phase(crystal_frame=CRYSTAL)
    for fixture_id in ("fe_bcc", "ni_fcc", "diamond", "zr_hcp")
}

print(f"{'fixture':<10} {'phase':<18} {'space group':<12} {'number':>7} {'centring':>9} "
      f"{'sites':>6}")
for fixture_id, phase in PHASES.items():
    condition = ReflectionCondition.from_phase(phase)
    print(f"{fixture_id:<10} {phase.name:<18} {str(phase.space_group_symbol):<12} "
          f"{phase.space_group_number:>7} {condition.centering:>9} "
          f"{len(phase.unit_cell.sites):>6}")

## 1. A phase is four objects, and each answers a different question

| component | what it is | what it answers |
| --- | --- | --- |
| `Lattice` | six cell parameters, hence the metric tensor | lengths, angles, $d$-spacings, volumes |
| `SymmetrySpec` | the proper rotation group | orientations, families, fundamental sectors |
| `UnitCell` | the atomic basis, as fractional coordinates | structure factors, hence *intensities* |
| `SpaceGroupSpec` | symbol and number | centring, and the translational symmetry the point group cannot carry |

Tutorials 01 and 03 used only the first two. The interesting content of this notebook is entirely in
the third and fourth, because those are what make a *crystal* out of a lattice.

The lattice half is quick, and worth one exact check: closed-form cell volumes.

In [ ]:
zirconium = PHASES["zr_hcp"]
lattice = zirconium.lattice
print(f"{zirconium.name}: a = {lattice.a:.4f} A, c = {lattice.c:.4f} A, "
      f"gamma = {lattice.gamma_deg:.1f} deg, c/a = {lattice.c / lattice.a:.4f}")

volume_from_metric = float(np.sqrt(np.linalg.det(metric_tensor(lattice))))
volume_closed_form = np.sqrt(3.0) / 2.0 * lattice.a**2 * lattice.c
print(f"\nsqrt(det g)                 : {volume_from_metric:.6f} A^3")
print(f"closed form (sqrt3/2) a^2 c : {volume_closed_form:.6f} A^3")
assert np.isclose(volume_from_metric, volume_closed_form)

nickel = PHASES["ni_fcc"]
cubic_volume = float(np.sqrt(np.linalg.det(metric_tensor(nickel.lattice))))
print(f"\n{nickel.name}: a = {nickel.lattice.a:.5f} A, "
      f"sqrt(det g) = {cubic_volume:.6f} A^3, a^3 = {nickel.lattice.a**3:.6f} A^3")
assert np.isclose(cubic_volume, nickel.lattice.a**3)

planes = MillerPlaneSet.from_hkl(np.array([[1, 1, 1], [2, 0, 0], [2, 2, 0]]), phase=nickel)
analytic = nickel.lattice.a / np.sqrt(np.sum(np.asarray(planes.indices) ** 2, axis=1))
print(f"\nnickel d-spacings   : {np.round(np.asarray(planes.d_spacings_angstrom()), 5)}")
print(f"a / sqrt(h^2+k^2+l^2): {np.round(analytic, 5)}")
assert np.allclose(planes.d_spacings_angstrom(), analytic)

## 2. The atomic basis, and what "site count" means

The `UnitCell` holds fractional coordinates in the *crystal* basis — not Cartesian, and not
normalizable, for the reasons tutorial 01 measured. The number of sites is a property of the
**conventional cell**, which is not the formula unit and not the primitive cell.

In [ ]:
for fixture_id in ("fe_bcc", "ni_fcc", "diamond", "zr_hcp"):
    phase = PHASES[fixture_id]
    metadata = get_phase_fixture(fixture_id).metadata
    print(f"{phase.name} ({phase.space_group_symbol}): {len(phase.unit_cell.sites)} sites in the "
          f"conventional cell, {metadata['expected_primitive_site_count']} in the primitive cell")
    for site in phase.unit_cell.sites:
        print(f"    {site.label:<5} {site.species:<3} "
              f"{np.round(site.fractional_coordinates, 5)}  occupancy {site.occupancy:.2f}")
    assert len(phase.unit_cell.sites) == metadata["expected_conventional_cell_site_count"]
    print()

Nickel has four sites and one atom per primitive cell: the conventional face-centred cubic
cell contains four lattice points, and the "extra" three are the centring translations, not extra
atoms in any physical sense. Diamond has eight, which is the same four centring translations times a
two-atom motif. Zirconium's two sites at $(\tfrac13,\tfrac23,\tfrac14)$ and
$(\tfrac23,\tfrac13,\tfrac34)$ are a genuine two-atom motif in a *primitive* hexagonal cell — and
those two coordinates are the entire reason hcp has the selection rule it has.

## 3. The structure factor is a lattice sum

$$F_{hkl} = \sum_j o_j f_j(s)\, e^{-B_j s^2}\, e^{2\pi i (h x_j + k y_j + l z_j)},
\qquad s = \frac{|\mathbf{g}|}{2}$$

with $o_j$ the occupancy, $f_j$ the atomic scattering factor and $B_j$ the Debye–Waller factor. For a
monatomic crystal $f$ is common to every term, so it factors out and what remains is purely
geometric:

$$F_{hkl} = f \sum_j e^{2\pi i (h x_j + k y_j + l z_j)}
        \;\equiv\; f\, G_{hkl}.$$

$G_{hkl}$ depends only on where the atoms sit, and **its zeros are the systematic absences**. So the
selection rules are computable from the fractional coordinates alone, with no scattering physics at
all — which is exactly what the next cell does.

> **Algorithm — the geometric structure factor**
>
> 1. Stack the fractional coordinates into an $(n_{\text{sites}}, 3)$ array.
> 2. Contract with the index triples: $\Phi = \mathbf{h}\,X^{\mathsf{T}}$, one matrix product for
>    every reflection and every site at once.
> 3. Sum $o_j e^{2\pi i \Phi}$ over sites.
> 4. A reflection is systematically absent when $|G| = 0$ — an *exact* zero from cancelling roots of
>    unity, not a small number.

In [ ]:
def geometric_factor(phase, indices):
    sites = phase.unit_cell.sites
    fractional = np.asarray([site.fractional_coordinates for site in sites], dtype=np.float64)
    occupancy = np.asarray([site.occupancy for site in sites], dtype=np.float64)
    phases = np.asarray(indices, dtype=np.float64) @ fractional.T
    return np.abs(np.sum(occupancy * np.exp(2j * np.pi * phases), axis=-1))


REFLECTIONS = np.array([
    [1, 0, 0], [1, 1, 0], [1, 1, 1], [2, 0, 0], [2, 1, 0], [2, 1, 1],
    [2, 2, 0], [2, 2, 2], [3, 1, 1], [3, 3, 1], [0, 0, 1], [0, 0, 2], [0, 0, 3],
])

print(f"{'hkl':<10} {'bcc Fe':>8} {'fcc Ni':>8} {'diamond':>9} {'hcp Zr':>8}")
table = {name: geometric_factor(PHASES[name], REFLECTIONS)
         for name in ("fe_bcc", "ni_fcc", "diamond", "zr_hcp")}
for row, indices in enumerate(REFLECTIONS):
    values = "".join(f"{table[name][row]:>9.3f}" for name in table)
    print(f"{str(tuple(int(v) for v in indices)):<10}{values}")

Every classical rule is in that table, and none of it was told to the code.

**Body-centred iron.** Sites at $(0,0,0)$ and $(\tfrac12,\tfrac12,\tfrac12)$, so
$G = 1 + e^{i\pi(h+k+l)}$, which is $2$ for $h+k+l$ even and $0$ for odd. Read the column: $(110)$,
$(200)$, $(211)$, $(222)$ present; $(100)$, $(111)$, $(210)$, $(311)$ absent.

**Face-centred nickel.** Four sites give $G = 4$ when $h$, $k$, $l$ share a parity and $0$
otherwise: $(111)$, $(200)$, $(220)$, $(311)$, $(331)$ present; $(110)$, $(210)$, $(211)$ absent.

**Diamond.** The same face-centring times a two-atom motif at $(0,0,0)$ and
$(\tfrac14,\tfrac14,\tfrac14)$, which contributes $1 + e^{i\pi(h+k+l)/2}$ — and that factor
vanishes when $h+k+l \equiv 2 \pmod 4$. Hence the $\mathbf{(222)}$ **is absent** even though it
passes the fcc test, and $(111)$ comes out at $4\sqrt2 = 5.657$ rather than $8$ because the two
sub-lattices are in quadrature.

**Hexagonal zirconium.** Two sites give
$G = |e^{2\pi i(h/3 + 2k/3 + l/4)} + e^{2\pi i(2h/3 + k/3 + 3l/4)}|$. For $(0001)$ that is
$|i + (-i)| = 0$, and for $(0002)$ it is $|-1 + -1| = 2$. So the first-order basal reflection is
extinguished and the second is the strongest thing in the pattern.

In [ ]:
# The rules, stated as predicates and checked against the sum for every reflection
# in a large index range.
rng = np.random.default_rng(3)
grid = np.stack(np.meshgrid(*[np.arange(-4, 5)] * 3, indexing="ij"), axis=-1).reshape(-1, 3)
grid = grid[np.any(grid != 0, axis=1)]
h, k, l = grid.T

RULES = {
    "fe_bcc": (h + k + l) % 2 == 0,
    "ni_fcc": ((h % 2 == 0) & (k % 2 == 0) & (l % 2 == 0))
    | ((np.abs(h) % 2 == 1) & (np.abs(k) % 2 == 1) & (np.abs(l) % 2 == 1)),
    "diamond": (
        (((h % 2 == 0) & (k % 2 == 0) & (l % 2 == 0)) & ((h + k + l) % 4 != 2))
        | ((np.abs(h) % 2 == 1) & (np.abs(k) % 2 == 1) & (np.abs(l) % 2 == 1))
    ),
    "zr_hcp": ~(((h + 2 * k) % 3 == 0) & (l % 2 == 1)),
}
print(f"{'structure':<10} {'reflections':>12} {'predicted present':>18} {'|G| > 0':>9} "
      f"{'rule matches sum':>18}")
for fixture_id, predicted in RULES.items():
    magnitude = geometric_factor(PHASES[fixture_id], grid)
    present = magnitude > 1e-9
    print(f"{fixture_id:<10} {len(grid):>12} {int(predicted.sum()):>18} {int(present.sum()):>9} "
          f"{str(bool(np.array_equal(predicted, present))):>18}")
    assert np.array_equal(predicted, present), fixture_id

Seven hundred and twenty-eight reflections per structure, and the textbook predicate agrees with the
lattice sum for every single one, in all four structures. The rules are not independent facts; they
are a compressed description of where the roots of unity cancel.

> **The awe note.** The diamond $(222)$ is the most interesting entry in that table, because it is
> forbidden and it is *observed*. The lattice sum says exactly zero: the two carbon sub-lattices are
> in antiphase for $h+k+l \equiv 2 \pmod 4$. But a real diamond gives a weak $(222)$, and it does so
> because the sum above assumes spherically symmetric atoms. Covalent bonding puts charge *between*
> the atoms, in the tetrahedral directions, which breaks the assumed symmetry and lets the
> cancellation fail slightly. The forbidden reflection is therefore a direct measurement of the
> bonding charge density — first done by Renninger in 1937, and still the textbook example of a
> "forbidden" reflection that carries information precisely because it should not be there. What
> PyTex computes here is the spherical-atom prediction, and its exact zero is the baseline against
> which that effect is measured.

## 4. Why a centring predicate is not the selection rule

`ReflectionCondition` implements the *centring* test — the part of the rule that comes from the
lattice type, $F$, $I$, $A$, $B$, $C$, $R$ or $P$. It is fast and it is what the simulation surfaces
use to prune candidates. It is not the whole rule, because the space group also carries screw axes
and glide planes, whose absences are invisible to it.

Zirconium is the clean demonstration: $P6_3/mmc$ is a **primitive** lattice, so the centring test
forbids nothing at all, and yet $(0001)$ is extinguished by the $6_3$ screw axis.

In [ ]:
basal = np.array([[0, 0, 1], [0, 0, 2], [0, 0, 3], [0, 0, 4], [1, 1, 1], [1, 1, 2]])
condition = ReflectionCondition.from_phase(zirconium)
allowed = np.asarray(centering_allowed_mask(basal, condition))
magnitude = geometric_factor(zirconium, basal)

print(f"zirconium: space group {zirconium.space_group_symbol}, centring "
      f"'{condition.centering}' (primitive)\n")
print(f"{'hkl':<10} {'centring allows':>16} {'|G|':>8} {'actually present':>18}")
for indices, permitted, value in zip(basal, allowed, magnitude):
    print(f"{str(tuple(int(v) for v in indices)):<10} {str(bool(permitted)):>16} {value:>8.3f} "
          f"{str(bool(value > 1e-9)):>18}")
print("\nThe centring test permits every one of these. The structure factor extinguishes")
print("(0001), (0003) and (1121) -- the 6_3 screw axis and the c glide, which a lattice-type")
print("predicate cannot see.")
assert bool(allowed[0]) and magnitude[0] < 1e-9

In [ ]:
# And the honest scale of the discrepancy over a whole index range.
print(f"{'structure':<10} {'centring allows':>16} {'|G| > 0':>9} {'centring over-counts by':>25}")
for fixture_id, phase in PHASES.items():
    permitted = np.asarray(
        centering_allowed_mask(grid, ReflectionCondition.from_phase(phase))
    ).sum()
    present = int((geometric_factor(phase, grid) > 1e-9).sum())
    print(f"{fixture_id:<10} {int(permitted):>16} {present:>9} "
          f"{int(permitted) - present:>25}")
print("\nZero for the two simple metals, whose only absences *are* the centring ones.")
print("Large for diamond and for hcp, where the space group carries more than a lattice type.")

For iron and nickel the two agree exactly, which is why the shortcut is safe there and why it is easy
to believe it is general. For diamond it over-counts by hundreds of reflections, and for hcp by
hundreds more. A simulation that prunes on centring alone and stops there will draw spots the crystal
does not produce.

## 5. Real scattering factors, and where the absolute scale comes in

The geometric factor is dimensionless and structure-only. For intensities the atomic scattering
factor has to be there, and for electrons that means Mott–Bethe:

$$f_e(s) = \frac{Z - f_x(s)}{8\pi^2 a_0 s^2},$$

in ångström, with the relativistic factor $\gamma = 1 + E/m_0c^2$ applied because the incident
electron is not slow. `electron_structure_factor_angstrom` does this and returns $F_g$ on an
*absolute* scale, which is what extinction distances and dynamical calculations need.

In [ ]:
strong = np.array([[1, 1, 1], [2, 0, 0], [2, 2, 0], [3, 1, 1], [2, 2, 2], [4, 0, 0]])
electron_factors = np.abs(electron_structure_factor_angstrom(nickel, strong, beam_energy_kev=200.0))
geometry_only = geometric_factor(nickel, strong)
spacings = np.asarray(MillerPlaneSet.from_hkl(strong, phase=nickel).d_spacings_angstrom())

print(f"nickel at 200 kV")
print(f"{'hkl':<10} {'d (A)':>8} {'|G| (geometry)':>16} {'|F_g| (A)':>11} {'|F_g| / |G|':>12}")
for indices, spacing, geometric, electron in zip(strong, spacings, geometry_only, electron_factors):
    ratio = electron / geometric if geometric > 1e-9 else float("nan")
    print(f"{str(tuple(int(v) for v in indices)):<10} {spacing:>8.4f} {geometric:>16.3f} "
          f"{electron:>11.4f} {ratio:>12.4f}")
print("\nThe last column is gamma f_e(s), and it falls with s -- which is the whole reason")
print("high-index reflections are weak even when they are allowed. The geometric factor is")
print("flat at 4 for every allowed fcc reflection and carries none of that.")
assert electron_factors[0] > electron_factors[-1]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.2))

# Panel 1: the selection rules as a map over one reciprocal-lattice layer.
layer = np.stack(np.meshgrid(np.arange(-6, 7), np.arange(-6, 7), indexing="ij"), axis=-1)
layer = np.concatenate([layer.reshape(-1, 2), np.zeros((layer.size // 2, 1))], axis=1)
layer = layer[np.any(layer != 0, axis=1)]
for offset, (fixture_id, marker, colour) in enumerate(
    (("fe_bcc", "o", "#1f77b4"), ("ni_fcc", "s", "#d62728"))
):
    present = geometric_factor(PHASES[fixture_id], layer) > 1e-9
    axes[0].scatter(layer[present, 0], layer[present, 1], s=34 - 14 * offset, marker=marker,
                    facecolor="none" if offset else colour, edgecolor=colour,
                    label=f"{fixture_id} allowed", linewidth=1.2)
axes[0].set_xlabel("h"), axes[0].set_ylabel("k")
axes[0].set_title("the (hk0) layer: which reflections survive")
axes[0].set_aspect("equal"), axes[0].legend(fontsize=8)

# Panel 2: allowed reflections against d-spacing for all four structures.
for fixture_id, colour in zip(PHASES, ("#1f77b4", "#d62728", "#2ca02c", "#ff7f0e")):
    phase = PHASES[fixture_id]
    magnitude = geometric_factor(phase, grid)
    present = magnitude > 1e-9
    spacings_all = np.asarray(
        MillerPlaneSet.from_hkl(grid[present], phase=phase).d_spacings_angstrom()
    )
    axes[1].hist(spacings_all, bins=np.linspace(0.3, 3.6, 60), histtype="step",
                 color=colour, label=phase.name, linewidth=1.3)
axes[1].set_xlabel(r"$d$ (Å)"), axes[1].set_ylabel("allowed reflections")
axes[1].set_yscale("log")
axes[1].set_title("allowed reflections to index 4")
axes[1].legend(fontsize=7.5)
fig.tight_layout()

The left panel is the selection rule as a picture: in the $(hk0)$ layer body-centring keeps $h+k$
even and face-centring keeps $h$ and $k$ both even, so the fcc set is a sublattice of the bcc set. The
right panel shows the consequence for a powder pattern — the number of lines available above any
given $d$ differs by a factor of a few between structures with the *same* lattice type.

## 6. Provenance: what a pinned fixture buys

Every fixture carries the source database, the record identifier, the citation, and a SHA-256 of the
CIF. The reason is that a structure downloaded live is not reproducible: databases are revised, and a
notebook whose numbers came from "the current CIF" cannot be checked a year later.

In [ ]:
record = get_phase_fixture("zr_hcp")
metadata = record.metadata
print(f"fixture         : {record.fixture_id}")
print(f"source          : {metadata['source_family']} record {metadata['source_record_id']}")
print(f"citation        : {metadata['citation']}")
print(f"redistribution  : {metadata['redistribution'][:60]}...")
print(f"CIF sha256      : {record.artifact_sha256[:32]}...")
print(f"lattice in file : a = {metadata['lattice_parameters_angstrom']['a']}, "
      f"c = {metadata['lattice_parameters_angstrom']['c']} A")
print(f"loaded lattice  : a = {zirconium.lattice.a}, c = {zirconium.lattice.c} A")
assert np.isclose(zirconium.lattice.a, metadata["lattice_parameters_angstrom"]["a"])
assert np.isclose(zirconium.lattice.c, metadata["lattice_parameters_angstrom"]["c"])

print(f"\nprovenance carried on the loaded phase:")
print(f"  source system : {zirconium.provenance.source_system}")
print(f"  reader        : {dict(zirconium.provenance.metadata)}")

## 7. Failure modes, deliberately triggered

**(a) A phase with no declared space group is simulated as primitive.** `ReflectionCondition` reads
the centring from the first letter of the symbol and falls back to $P$ when there is no symbol at
all. That is the only defensible default, and it is silently wrong for a centred structure — so PyTex
provides a predicate that makes the situation visible rather than leaving it to be discovered.

In [ ]:
from pytex import Phase

bare = Phase(
    "nickel-no-symbol",
    lattice=nickel.lattice,
    symmetry=nickel.symmetry,
    crystal_frame=CRYSTAL,
    unit_cell=nickel.unit_cell,
)
print(f"{'phase':<20} {'centring declared?':>19} {'centring used':>14}")
for phase in (nickel, bare):
    print(f"{phase.name:<20} {str(phase_centering_is_declared(phase)):>19} "
          f"{ReflectionCondition.from_phase(phase).centering:>14}")

check = np.array([[1, 1, 0], [2, 1, 0], [1, 0, 0]])
print(f"\n{'hkl':<10} {'declared F allows':>18} {'assumed P allows':>18} {'|G|':>8}")
for indices, with_symbol, without_symbol, value in zip(
    check,
    np.asarray(centering_allowed_mask(check, ReflectionCondition.from_phase(nickel))),
    np.asarray(centering_allowed_mask(check, ReflectionCondition.from_phase(bare))),
    geometric_factor(nickel, check),
):
    print(f"{str(tuple(int(v) for v in indices)):<10} {str(bool(with_symbol)):>18} "
          f"{str(bool(without_symbol)):>18} {value:>8.3f}")
print("\nWithout the symbol the pruning admits every reflection, and the pattern gains spots")
print("that the structure factor would have removed. Check phase_centering_is_declared before")
print("trusting a reflection list.")
assert not phase_centering_is_declared(bare)

**(b) Reading the site count as the formula unit.** Nickel's conventional cell has four sites and one
atom per primitive cell. A composition computed from conventional-cell site counts is right only if
the multiplicity of every species is scaled the same way — which it is for a monatomic metal and is
not, in general, for a compound.

In [ ]:
print(f"{'phase':<18} {'conventional':>13} {'primitive':>10} {'ratio':>7} {'formula':>9}")
for fixture_id, phase in PHASES.items():
    metadata = get_phase_fixture(fixture_id).metadata
    conventional = metadata["expected_conventional_cell_site_count"]
    primitive = metadata["expected_primitive_site_count"]
    print(f"{phase.name:<18} {conventional:>13} {primitive:>10} "
          f"{conventional // primitive:>7} {str(metadata['chemical_formula']):>9}")
print("\nThe ratio is the number of lattice points in the conventional cell: 2 for I, 4 for F,")
print("1 for P. It is a property of the centring, not of the chemistry.")

**(c) Treating fractional coordinates as Cartesian.** They are components in the crystal basis, so in
a hexagonal cell the Cartesian distance between two sites is not the Euclidean distance between their
fractional triples.

In [ ]:
first, second = [site.fractional_coordinates for site in zirconium.unit_cell.sites]
difference = np.asarray(second) - np.asarray(first)
naive = float(np.linalg.norm(difference))
true = float(np.sqrt(difference @ metric_tensor(zirconium.lattice) @ difference))
print(f"the two zirconium sites: {np.round(first, 5)} and {np.round(second, 5)}")
print(f"\nEuclidean distance between the fractional triples: {naive:.4f}  (dimensionless)")
print(f"actual interatomic distance                      : {true:.4f} A")
print(f"nearest-neighbour distance in ideal hcp, = a      : {zirconium.lattice.a:.4f} A")
print("\nThe first number is not a length and has no units. The second is the physical")
print("separation, and it is close to a because zirconium is a nearly ideal hcp metal.")
assert abs(true - zirconium.lattice.a) < 0.05 * zirconium.lattice.a

> **Good to know.**
>
> - The first crystal structure ever solved was rock salt, by W. H. and W. L. Bragg in 1913, and
>   the answer was chemically shocking: there is no NaCl *molecule* in the crystal, only an
>   alternating lattice of ions. Chemists disputed it for years.
> - W. L. Bragg was 25 when he shared the 1915 Nobel Prize with his father, and remains the
>   youngest science laureate.
> - The structure factor is why extinctions are a *phase* property rather than a lattice one: fcc
>   forbids mixed-parity reflections whatever the atoms are, but NaCl's weak 111 comes from
>   Na$^{+}$ and Cl$^{-}$ scattering almost out of phase. One rule is arithmetic on indices; the
>   other needs the basis and the scattering factors.

## 8. What this implementation does not do

- **`ReflectionCondition` is a centring test only.** Screw-axis and glide-plane absences come from
  the structure factor, and section 4 measures the difference. There is no general space-group
  reflection-condition table.
- **No symmetry-operation expansion from the space group.** The pinned CIFs list every site of the
  conventional cell explicitly. A CIF that gives only the asymmetric unit plus a space-group symbol
  is expanded by pymatgen on import, not by PyTex.
- **Occupancies and Debye–Waller factors are carried, not fitted.** `b_iso` is `None` on these
  fixtures, so no thermal damping is applied and the computed factors are the zero-temperature,
  spherical-atom values.
- **Spherical atoms.** The awe note's whole point: the structure factor assumes spherically
  symmetric charge, so genuinely forbidden reflections come out exactly zero and the bonding-density
  effects that make them observable are not modelled.
- **One phase, one lattice.** Modulated structures, superlattices and incommensurate phases are out
  of scope; `Phase` is a periodic crystal with one cell.

## 9. What to take away

- **A `Phase` is four objects.** Lattice for geometry, symmetry for orientations, unit cell for
  intensities, space group for translational symmetry. Notebooks that need only the first two do not
  need a CIF at all.
- **The selection rules are the zeros of one lattice sum.** All four classical rules — bcc, fcc,
  diamond, hcp — reproduced here from fractional coordinates alone, agreeing with the textbook
  predicates for 728 reflections each.
- **A centring predicate is a shortcut, not the rule.** Exact for simple metals, and over-counting by
  hundreds of reflections for diamond and hcp, because a screw axis extinguishes reflections in a
  primitive lattice.
- **The geometric factor is flat; the physics is in $f_e(s)$.** $|G| = 4$ for every allowed fcc
  reflection, so the fall-off with $d$ comes entirely from the scattering factor.
- **Check `phase_centering_is_declared`.** A phase without a space-group symbol is treated as
  primitive, which is silently wrong for a centred structure.
- **Fractional coordinates are crystal-basis components.** Their Euclidean difference is not a
  distance.

### Further reading

- Tutorial 01, *Reference frames* — the metric tensor used for every distance and volume here.
- Tutorial 03, *Crystal symmetry* — the point group, and why it cannot carry the translational
  symmetry that produces screw-axis absences.
- Tutorial 11, *Powder XRD workflows* — the same structure factors turned into intensities, with
  multiplicity and the Lorentz–polarization factor.
- Tutorial 12, *SAED workflows* — where the absences of section 4 become visible as missing spots,
  and where double diffraction puts some of them back.
- `docs/site/theory/crystal_structures_and_cif_import.md` — the import path, the provenance contract,
  and the site-count validation.
- *International Tables for Crystallography*, Vol. A, 6th ed. (Wiley, 2016) — the space groups and
  their reflection conditions.
- M. Renninger, *Z. Phys.* **106** (1937) 141 — the observation of the "forbidden" diamond $(222)$.
- B. E. Warren, *X-ray Diffraction* (Dover, 1990), Ch. 3 — the structure factor as a lattice sum, with
  the four rules derived the way section 3 computes them.